In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from transformers import AutoTokenizer
import torch
from openai import OpenAI

In [ ]:
import json
import csv
import re
import math
import numpy as np
import pandas as pd
import asyncio
import matplotlib.pyplot as plt 

from pydantic import BaseModel
from enum import Enum

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

! nvidia-smi


In [ ]:
from huggingface_hub import login
HF_TOKEN = ""
login(HF_TOKEN)  # This logs you in for the session

import requests
response = requests.get("https://huggingface.co")
print(response.status_code)  # Should print 200 if the connection is successful

model_name = "neuralmagic/Meta-Llama-3.1-70B-Instruct-quantized.w4a16"

In [ ]:
llm = LLM(model=model_name, device=device, max_model_len=65536, tensor_parallel_size=1, enable_prefix_caching=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

class DrugStopJSON(BaseModel):
    drug_stop_phrase: str
    reason_for_stopping: str
    drug_name: str

json_schema = DrugStopJSON.model_json_schema()

guided_decoding_params = GuidedDecodingParams(json=json_schema)
sampling_params = SamplingParams(guided_decoding=guided_decoding_params, temperature=0, max_tokens=16384)
# sampling_params = SamplingParams(temperature=0.5, max_tokens=1000)

# Structure the messages list
system_prompt = {
  "role": "system", "content": """You are a clinical language model specialized in Estonian medical documentation. You are given clinical summary charts written by doctors about their patients who stopped taking their medication. Your task is to extract the reasons why the patients stopped taking the medications.
Return your output as a JSON object with the keys: "drug_stop_phrase", "reason_for_stopping", "drug_name".

Extraction rules:
"drug_stop_phrase": Must include the full sentence(s) that should contain the fact that the patient stopped taking the medication and the reason, if they exist. Include multiple sentences if needed to preserve semantic clarity.

"reason_for_stopping": Extract only the reason (e.g., "vererõhk normaliseerus", "tekkis köha", "ei pidanud vajalikuks"). 

"drug_name": Extract the name of the drug or its type/class only if explicitly mentioned (e.g., "Nebilet", "Amlodipin", "vererõhuravim", “diabeediravim”, “statiin”).

Output format:
Return only one JSON object per matched case.
Do not paraphrase or clean the extracted texts; return the exact text as-is from the input.
If no valid case is found in the input for one or more of the classes, leave the value as an empty string.
"""
}

messages_template = [
    system_prompt,
    # Example 1
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},

    # Example 2
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},

    # Example 3
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},

    # Example 4
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},

    # Example 5
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},

    # Example 6
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},
    
    # Example 7
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},
    
    # Example 8
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},
    
    # Example 9
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},

    # Example 10
    {"role": "user", "content": """"""},
    {"role": "assistant", "content": "{\"drug_stop_phrase\": \"\", \"reason_for_stopping\": \"\", \"drug_name\": \"\"}"},

]


In [ ]:
texts_file = "a10_switches_original.csv"

df = pd.read_csv(texts_file)  # texts_file should be the path to your CSV file
nr_of_texts = len(df)

# Extract the "anamnesis" column into a list
texts = df["anamnesis"].dropna().tolist()[:nr_of_texts]  # Drop NaN values if any
ids = df["id"].dropna().tolist()[:nr_of_texts]

print(nr_of_texts)

In [ ]:
messages_list = [
    messages_template + [{"role": "user", "content": text}]
    for text in texts # only first 100 texts, beware and change
]

# Tokenize all messages in one go
prompts = [
    tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    for messages in messages_list
]
print(len(prompts))

# Output for all texts at once (progress bar seems to be bugged)
outputs = llm.generate(
    prompts=prompts,  # Pass messages instead of a raw string
    sampling_params=sampling_params,
)

In [ ]:
output_dir = ""
run_name = "a10_switch_stops"

output_file = output_dir + run_name + ".csv"

output_list = [output.outputs[0].text for output in outputs]

with open(output_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file, quoting=csv.QUOTE_MINIMAL)
    
    # Writing header
    writer.writerow(["id", "result"])  
    
    # Convert dictionary to JSON string before writing
    writer.writerows(zip(ids, (json.dumps(entry, ensure_ascii=False) for entry in output_list)))

print(f"CSV file '{output_file}' has been created successfully.")